In [2]:
import zenodopy
import os
from urllib.request import urlretrieve
import geopandas as gpd
import pandas as pd

In [3]:
DOI = "10.5281/zenodo.15261836"
OUT_DIR = "../data/spread_data/"

In [12]:
#pull data via zenodopy - a Zenodo API key is needed. Further info on setup here: https://github.com/lgloege/zenodopy
#Zenodo archive is here: https://zenodo.org/records/15261836
def pull_fire_spread_data():
    zeno = zenodopy.Client()
    content = zeno.get_urls_from_doi(DOI)

    os.makedirs(OUT_DIR, exist_ok=True)

    for url in content:
        basenm = url.split("/")[-2]
        filename = os.path.join(OUT_DIR, basenm)
        urlretrieve(url, filename)


zeno = zenodopy.Client()
pull_fire_spread_data()


In [4]:
# Load NERC regions - Code taken from ./notebooks/California_shape_files.ipynb
regions_gdf = gpd.read_file("../data/NERC_Reliability_Coordinators.geojson")
# Ensure the CRS is WGS84 (EPSG:4326)
if regions_gdf.crs is None or regions_gdf.crs.to_string() != "EPSG:4326":
    print('converting to EPSG: 4326')
    regions_gdf = regions_gdf.to_crs("EPSG:4326")
CAISO = regions_gdf[regions_gdf['NAME']=='CALIFORNIA INDEPENDENT SYSTEM OPERATOR']
print(CAISO.keys())
print(len(CAISO['geometry']), CAISO['geometry'])

Index(['OBJECTID', 'ID', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP',
       'TELEPHONE', 'COUNTRY', 'SOURCE', 'SOURCEDATE', 'VAL_METHOD',
       'VAL_DATE', 'WEBSITE', 'ABBRV', 'SHAPE_Length', 'SHAPE_Area',
       'geometry'],
      dtype='object')
1 0    MULTIPOLYGON (((-117.21222 32.77345, -117.2125...
Name: geometry, dtype: geometry


In [5]:
#Load ROI for March 2023 event
mar23_roi = gpd.read_file("../data/selected_states_MarchApril2023_event.geojson")
print(mar23_roi.keys(), len(mar23_roi))
if mar23_roi.crs is None or mar23_roi.crs.to_string() != "EPSG:4326":
    print('converting to EPSG: 4326')
    mar23_roi = mar23_roi.to_crs("EPSG:4326")

Index(['REGION', 'DIVISION', 'STATEFP', 'STATENS', 'GEOID', 'GEOIDFQ',
       'STUSPS', 'NAME', 'LSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object') 14
converting to EPSG: 4326


In [6]:
def row_intersects(row, roi):
    intersects = False
    for idx, roi_row in roi.iterrows():
        intersects = intersects or row['geometry'].intersects(roi_row['geometry'])
        if intersects:
            break
    return intersects
def clip_fires_to_roi(fire_df, roi, intersection_key):
    #Find fires that ROI
    fire_df[intersection_key] = False
     
    fire_df[intersection_key] = fire_df.apply(row_intersects, roi=roi, axis=1)

    #subset and save to disk
    fire_subset = fire_df[fire_df[intersection_key] == True]
    print(len(fire_subset), len(fire_df))
    return fire_subset
 
def clip_fires_to_roi_initial_ds(roi, intersection_key):
    #Full table/key info here: https://www.nature.com/articles/s41597-022-01343-0 
    #This is file for 2014. All years (2012 - 2021) follow same naming convention
    for i in range(2012,2022):
        fname = os.path.join(OUT_DIR, "feds_western_us_" + str(i) + "_af_postprocessed.parquet")
        fire_df = gpd.read_parquet(fname)
        #Update CRS
        fire_df['geometry'] = fire_df['hull']
        fire_df.set_geometry('geometry',inplace=True)
        fire_df = fire_df.to_crs('EPSG:4326')
        print(fire_df.keys())
        fire_subset = clip_fires_to_roi(fire_df, roi, intersection_key)
        out_fname = os.path.join(OUT_DIR, "feds_western_us_" + str(i) + "_af_postprocessed_" + intersection_key + ".parquet")
        fire_subset.to_parquet(out_fname, index=True)
        

clip_fires_to_roi_initial_ds(CAISO, 'caiso_intersections')
#TODO def clip_fires_roi_2023(mar23_roi, 'mar23_intersections')        


Index(['mergeid', 'invalid', 'ftype', 'n_pixels', 'n_newpixels', 'farea',
       'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'hull', 'fline', 'nfp', 'geometry'],
      dtype='object')
113477 166399
Index(['mergeid', 'invalid', 'ftype', 'n_pixels', 'n_newpixels', 'farea',
       'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'hull', 'fline', 'nfp', 'geometry'],
      dtype='object')
128802 176853
Index(['mergeid', 'invalid', 'ftype', 'n_pixels', 'n_newpixels', 'farea',
       'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'hull', 'fline', 'nfp', 'geometry'],
      dtype='object')
132833 203755
Index(['mergeid', 'invalid', 'ftype', 'n_pixels', 'n_newpixels', 'farea',
       'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'hull', 'fline', 'nfp', 'geometry'],
      dtype='object')
126117 206232
Index(['mergeid', 'invalid', 'ftype', 'n_pixels', 'n_newpixels', 'farea'